<a href="https://colab.research.google.com/github/dchav12/New-stable-/blob/main/NBA_Prop_Hit_Sheet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

# Get uploaded filename dynamically
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Raw Columns:")
print(df.columns.tolist())

df.head()

In [ ]:
import numpy as np
import re

# Promote first row to header
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)

df.columns = df.columns.astype(str).str.strip()

# Drop empty columns
df = df.loc[:, df.columns.notna()]
df = df.dropna(axis=1, how='all')

# Extract American odds from "BET\n-169"
def extract_american_odds(x):
    if pd.isna(x):
        return np.nan
    match = re.search(r'([+-]\d+)', str(x))
    return float(match.group(1)) if match else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Normalize percentage columns
pct_cols = ["IM PROB %","L5","L10","2025","HM/AW","H2H",
            "DVP L5","DVP L10","DVP 2025","DVP HM/AW"]

for c in pct_cols:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("%","", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

print("Cleaned Columns:")
print(df.columns.tolist())

df.head()

In [ ]:
 import numpy as np
import pandas as pd

# ---------- Helpers ----------
def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

def american_to_decimal(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return 1.0 + (100.0/(-o))
    return 1.0 + (o/100.0)

def kelly_fraction(p, dec_odds):
    # For decimal odds d: b = d - 1
    if pd.isna(p) or pd.isna(dec_odds):
        return np.nan
    b = dec_odds - 1.0
    q = 1.0 - p
    if b <= 0:
        return np.nan
    f = (b*p - q) / b
    return max(0.0, f)

def safe_mean(row, cols):
    vals = [row[c] for c in cols if c in row.index and pd.notna(row[c])]
    return np.mean(vals) if len(vals) else np.nan

# ---------- 1) Book probabilities ----------
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# ---------- 2) Weighted True Probability (NBA tuned) ----------
# NBA: more stable; avoid overfitting L5. Use L10/Season heavier.
W = {
    "L5": 0.20,
    "L10": 0.30,
    "2025": 0.25,
    "HM/AW": 0.10,
    "H2H": 0.05,
    "IM PROB %": 0.10,   # optional “market-like” signal; keeps us anchored
}

def weighted_prob(row):
    num, den = 0.0, 0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w * float(v)
            den += w
    return num/den if den > 0 else np.nan

df["P_TREND"] = df.apply(weighted_prob, axis=1)

# ---------- 3) DVP Matchup Layer (capped) ----------
# Use available dvp cols; compute mean and cap its influence so it can't dominate.
dvp_cols = [c for c in ["DVP L5","DVP L10","DVP 2025","DVP HM/AW"] if c in df.columns]
df["P_DVP_RAW"] = df.apply(lambda r: safe_mean(r, dvp_cols), axis=1)

def dvp_adjust(p_dvp):
    if pd.isna(p_dvp):
        return 0.0
    # centered around 0.50; cap impact to +/- 0.06 for NBA
    adj = float(p_dvp) - 0.50
    return float(np.clip(adj, -0.06, 0.06))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust)

# ---------- 4) Final Model Probability ----------
df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

# ---------- 5) Edge + EV ----------
df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"] * (df["DEC_ODDS"] - 1.0) - (1.0 - df["MODEL_PROB"])

# ---------- 6) Kelly sizing ----------
df["KELLY_FULL"] = df.apply(lambda r: kelly_fraction(r["MODEL_PROB"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF"] = 0.5 * df["KELLY_FULL"]

# Convert to "units" with a simple bankroll convention:
# If 1u = 1% bankroll, units = kelly_half * 100
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"] * 100).round(2)
df["KELLY_FULL_UNITS"] = (df["KELLY_FULL"] * 100).round(2)

# ---------- 7) Outputs ----------
# Top 10 Most Likely (by MODEL_PROB)
top10_likely = df.sort_values("MODEL_PROB", ascending=False).head(10)

# Top 10 Highest EV (by EV_1U)
top10_ev = df.sort_values("EV_1U", ascending=False).head(10)

print("TOP 10 MOST LIKELY")
display(top10_likely[[
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"
]])

print("\nTOP 10 HIGHEST EV")
display(top10_ev[[
    "PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS",
    "MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"
]])

# Save files
df.to_csv("nba_prop_engine_v1_full_output.csv", index=False)
top10_likely.to_csv("nba_prop_engine_v1_top10_most_likely.csv", index=False)
top10_ev.to_csv("nba_prop_engine_v1_top10_highest_ev.csv", index=False)

print("\nSaved: nba_prop_engine_v1_full_output.csv")
print("Saved: nba_prop_engine_v1_top10_most_likely.csv")
print("Saved: nba_prop_engine_v1_top10_highest_ev.csv")

In [ ]:
# =============================
# NBA PROP ENGINE — MASTER LOAD
# =============================

from google.colab import files
import pandas as pd
import numpy as np
import re

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

# Promote first row to header
df.columns = df.iloc[0]
df = df.drop(index=0).reset_index(drop=True)
df.columns = df.columns.astype(str).str.strip()

# Extract American odds
def extract_american_odds(x):
    if pd.isna(x):
        return np.nan
    match = re.search(r'([+-]\d+)', str(x))
    return float(match.group(1)) if match else np.nan

df["AMERICAN_ODDS"] = df["ODDS"].apply(extract_american_odds)

# Normalize % fields
pct_cols = ["IM PROB %","L5","L10","2025","HM/AW","H2H",
            "DVP L5","DVP L10","DVP 2025","DVP HM/AW"]

for c in pct_cols:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.replace("%","", regex=False)
        )
        df[c] = pd.to_numeric(df[c], errors="coerce") / 100.0

print("NBA data loaded and cleaned.")
print("Rows:", len(df))
df.head()

In [ ]:
# Drop junk columns created by blank header cells
df = df.loc[:, ~df.columns.isna()]  # drop NaN column names
df = df.loc[:, ~df.columns.astype(str).str.lower().isin(["nan"])]  # drop literal "nan"
df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]  # drop Unnamed: 0, etc.

# Strip column names again (safe)
df.columns = df.columns.astype(str).str.strip()

print("Columns now:", df.columns.tolist())
df.head()

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
from datetime import datetime, timezone

ODDS_API_KEY = "PASTE_YOUR_KEY_HERE"

def american_to_prob(odds):
    if pd.isna(odds):
        return np.nan
    o = float(odds)
    if o < 0:
        return (-o) / ((-o) + 100.0)
    return 100.0 / (o + 100.0)

def fetch_odds_api_nba_markets(regions="us", markets="player_points,player_rebounds,player_assists,player_threes,player_points_rebounds_assists"):
    url = "https://api.the-odds-api.com/v4/sports/basketball_nba/odds"
    params = {
        "apiKey": ODDS_API_KEY,
        "regions": regions,
        "markets": markets,
        "oddsFormat": "american",
        "dateFormat": "iso"
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

odds_json = fetch_odds_api_nba_markets()
print("Games returned:", len(odds_json))

In [ ]:
import numpy as np
import pandas as pd

def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return 1.0 + (100.0/(-o)) if o < 0 else 1.0 + (o/100.0)

def kelly_fraction(p, dec_odds):
    if pd.isna(p) or pd.isna(dec_odds): return np.nan
    b = dec_odds - 1.0
    q = 1.0 - p
    if b <= 0: return np.nan
    f = (b*p - q)/b
    return max(0.0, f)

# Book probs from your CSV odds
df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

# NBA weights (avoid overfitting L5)
W = {"L5":0.20, "L10":0.30, "2025":0.25, "HM/AW":0.10, "H2H":0.05, "IM PROB %":0.10}

def weighted_prob(row):
    num=den=0.0
    for k,w in W.items():
        v = row.get(k, np.nan)
        if pd.notna(v):
            num += w*float(v); den += w
    return num/den if den>0 else np.nan

df["P_TREND"] = df.apply(weighted_prob, axis=1)

# DVP capped (NBA)
dvp_cols = [c for c in ["DVP L5","DVP L10","DVP 2025","DVP HM/AW"] if c in df.columns]
df["P_DVP_RAW"] = df[dvp_cols].mean(axis=1, skipna=True)

def dvp_adjust(p):
    if pd.isna(p): return 0.0
    return float(np.clip(p - 0.50, -0.06, 0.06))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust)

df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.99)

df["EDGE"] = df["MODEL_PROB"] - df["BOOK_PROB"]
df["EV_1U"] = df["MODEL_PROB"]*(df["DEC_ODDS"]-1.0) - (1.0-df["MODEL_PROB"])

df["KELLY_FULL"] = df.apply(lambda r: kelly_fraction(r["MODEL_PROB"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF"] = 0.5*df["KELLY_FULL"]
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"]*100).round(2)

top10_likely = df.sort_values("MODEL_PROB", ascending=False).head(10)
top10_ev     = df.sort_values("EV_1U", ascending=False).head(10)

print("TOP 10 MOST LIKELY")
display(top10_likely[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","KELLY_HALF_UNITS"]])

print("\nTOP 10 HIGHEST EV")
display(top10_ev[["PLAYER","GAME","PROP","OUTCOME","AMERICAN_ODDS","MODEL_PROB","BOOK_PROB","EDGE","EV_1U","KELLY_HALF_UNITS"]])

df.to_csv("nba_prop_engine_v1_full_output.csv", index=False)
top10_likely.to_csv("nba_prop_engine_v1_top10_most_likely.csv", index=False)
top10_ev.to_csv("nba_prop_engine_v1_top10_highest_ev.csv", index=False)

print("\nSaved: nba_prop_engine_v1_full_output.csv")
print("Saved: nba_prop_engine_v1_top10_most_likely.csv")
print("Saved: nba_prop_engine_v1_top10_highest_ev.csv")

In [ ]:
import requests, re, numpy as np, pandas as pd
from functools import lru_cache

ESPN_NBA_SUMMARY  = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/summary"
ESPN_NBA_TEAMS    = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams"
ESPN_NBA_SCHEDULE = "https://site.api.espn.com/apis/site/v2/sports/basketball/nba/teams/{team_id}/schedule"

def get_json(url, params=None):
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

@lru_cache(maxsize=4096)
def get_team_list():
    return get_json(ESPN_NBA_TEAMS)

@lru_cache(maxsize=4096)
def get_schedule(team_id: str):
    return get_json(ESPN_NBA_SCHEDULE.format(team_id=str(team_id)))

@lru_cache(maxsize=8192)
def get_summary(event_id: str):
    return get_json(ESPN_NBA_SUMMARY, params={"event": str(event_id)})

def norm_name(s):
    s = str(s).upper().strip()
    s = re.sub(r"\b(JR|SR|II|III|IV)\b", "", s)
    s = re.sub(r"[^A-Z ]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def parse_minutes(mmss):
    if mmss is None or (isinstance(mmss,float) and np.isnan(mmss)):
        return np.nan
    s = str(mmss).strip()
    if ":" in s:
        m, sec = s.split(":", 1)
        try: return float(m) + float(sec)/60.0
        except: return np.nan
    try: return float(s)
    except: return np.nan

In [ ]:
# Build abbrev -> team_id map from ESPN
teams_json = get_team_list()

abbrev_to_id = {}
for sp in teams_json.get("sports", []):
    for lg in sp.get("leagues", []):
        for tm in lg.get("teams", []):
            t = tm.get("team", {})
            ab = (t.get("abbreviation") or "").upper()
            tid = t.get("id")
            if ab and tid:
                abbrev_to_id[ab] = str(tid)

print("ESPN teams mapped:", len(abbrev_to_id))
print("Example:", list(abbrev_to_id.items())[:10])

# Parse GAME into away/home abbrevs
def parse_game_abbrevs(game_str):
    s = str(game_str).upper().strip()
    if " AT " in s:
        away, home = s.split(" AT ", 1)
    elif " VS " in s:
        away, home = s.split(" VS ", 1)
    else:
        return None, None
    return away.strip(), home.strip()

df["AWAY_ABBR"], df["HOME_ABBR"] = zip(*df["GAME"].apply(parse_game_abbrevs))

df["AWAY_TEAM_ID"] = df["AWAY_ABBR"].map(abbrev_to_id)
df["HOME_TEAM_ID"] = df["HOME_ABBR"].map(abbrev_to_id)

missing_team = df[df["AWAY_TEAM_ID"].isna() | df["HOME_TEAM_ID"].isna()][["GAME","AWAY_ABBR","HOME_ABBR","AWAY_TEAM_ID","HOME_TEAM_ID"]].drop_duplicates()
print("Games with missing ESPN team ids:", len(missing_team))
display(missing_team.head(20))

In [ ]:
def completed_event_ids_from_schedule(team_id, max_games=15):
    j = get_schedule(team_id)
    events = j.get("events", []) or []
    evs = []

    for e in events:
        # event id
        eid = e.get("id") or None
        if eid is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            eid = e["competitions"][0].get("id")

        # status
        status = None
        if isinstance(e.get("status"), dict):
            status = (e["status"].get("type") or {}).get("name") or (e["status"].get("type") or {}).get("state")
        if status is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            st = (e["competitions"][0].get("status") or {}).get("type") or {}
            status = st.get("name") or st.get("state")

        st = str(status).upper() if status else ""
        if ("FINAL" not in st) and ("POST" not in st):
            continue

        dt = e.get("date")
        if dt is None and isinstance(e.get("competitions"), list) and len(e["competitions"]):
            dt = e["competitions"][0].get("date")

        evs.append({"eventId": str(eid), "date": dt})

    # sort most recent first
    evs = sorted(evs, key=lambda x: pd.to_datetime(x["date"], utc=True, errors="coerce"), reverse=True)
    evs = [x for x in evs if x["eventId"] not in ["None","nan",""]]
    return [x["eventId"] for x in evs[:max_games]]

team_ids = pd.unique(pd.concat([df["AWAY_TEAM_ID"], df["HOME_TEAM_ID"]]).dropna()).tolist()
team_recent_events = {}

for tid in team_ids:
    try:
        team_recent_events[str(tid)] = completed_event_ids_from_schedule(str(tid), max_games=15)
    except Exception as e:
        team_recent_events[str(tid)] = []
        print("Schedule fail team", tid, e)

print("Teams pulled:", len(team_recent_events))
print("Example counts:", {k: len(v) for k,v in list(team_recent_events.items())[:5]})

In [ ]:
def extract_minutes_for_player_in_event(event_id: str, player_norm: str):
    j = get_summary(event_id)
    box = j.get("boxscore", {})
    players = box.get("players", [])
    if not players:
        return np.nan

    for team_block in players:
        for stat_group in team_block.get("statistics", []):
            labels = stat_group.get("labels", [])
            if "MIN" not in labels:
                continue
            min_idx = labels.index("MIN")

            for a in stat_group.get("athletes", []):
                ath = a.get("athlete", {}) or {}
                nm = norm_name(ath.get("displayName",""))
                if nm != player_norm:
                    continue
                stats = a.get("stats", [])
                mm = stats[min_idx] if min_idx < len(stats) else None
                return parse_minutes(mm)

    return np.nan

# Build unique player keys from your sheet
df["PLAYER_NORM"] = df["PLAYER"].apply(norm_name)
unique_players = df[["PLAYER","PLAYER_NORM","AWAY_TEAM_ID","HOME_TEAM_ID"]].drop_duplicates()

N_BACK = 10
hist_rows = []

for _, r in unique_players.iterrows():
    pnorm = r["PLAYER_NORM"]
    away_tid = str(r["AWAY_TEAM_ID"]) if pd.notna(r["AWAY_TEAM_ID"]) else None
    home_tid = str(r["HOME_TEAM_ID"]) if pd.notna(r["HOME_TEAM_ID"]) else None

    candidate_events = []
    for tid in [away_tid, home_tid]:
        if tid and tid in team_recent_events:
            candidate_events += team_recent_events[tid][:N_BACK]

    # dedupe preserve order
    seen=set()
    candidate_events = [x for x in candidate_events if not (x in seen or seen.add(x))]
    candidate_events = candidate_events[:N_BACK]

    mins_list = []
    for eid in candidate_events:
        try:
            m = extract_minutes_for_player_in_event(eid, pnorm)
            if pd.notna(m):
                mins_list.append(m)
        except:
            continue

    if len(mins_list) == 0:
        hist_rows.append({"PLAYER_NORM": pnorm, "MIN_L5": np.nan, "MIN_L10": np.nan, "MIN_SD_L10": np.nan, "GAMES_FOUND": 0})
    else:
        arr = np.array(mins_list, dtype=float)
        hist_rows.append({
            "PLAYER_NORM": pnorm,
            "MIN_L5": float(np.nanmean(arr[:5])),
            "MIN_L10": float(np.nanmean(arr[:10])),
            "MIN_SD_L10": float(np.nanstd(arr[:10])) if len(arr[:10]) >= 2 else np.nan,
            "GAMES_FOUND": int(len(arr))
        })

mins_hist_df = pd.DataFrame(hist_rows)
print("Minutes rows:", len(mins_hist_df))
display(mins_hist_df.head(15))

# merge into df
df = df.merge(mins_hist_df, on="PLAYER_NORM", how="left")
print("Rows with GAMES_FOUND>0:", int((df["GAMES_FOUND"].fillna(0)>0).sum()), "of", len(df))

In [ ]:
def role_adj(min_l10, sd_l10):
    if pd.isna(min_l10):
        base = 0.0
    elif min_l10 >= 34:
        base = +0.030
    elif min_l10 >= 30:
        base = +0.015
    elif min_l10 >= 24:
        base = +0.005
    elif min_l10 >= 18:
        base = -0.020
    else:
        base = -0.045

    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.020
    elif sd_l10 >= 5:
        stab = -0.010
    else:
        stab = 0.0

    return base + stab

df["ADJ_ROLE"] = df.apply(lambda r: role_adj(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)

df["DNP_RATE_L10"] = (10 - df["GAMES_FOUND"].fillna(0)).clip(0,10) / 10.0
df["MIN_TREND_L5_L10"] = df["MIN_L5"] - df["MIN_L10"]

def inj_proxy(dnp, trend):
    pen = 0.0
    if pd.notna(dnp) and dnp >= 0.3: pen -= 0.02
    if pd.notna(dnp) and dnp >= 0.5: pen -= 0.04
    if pd.notna(trend) and trend <= -4: pen -= 0.02
    if pd.notna(trend) and trend <= -7: pen -= 0.04
    return pen

df["ADJ_INJ_PROXY"] = df.apply(lambda r: inj_proxy(r["DNP_RATE_L10"], r["MIN_TREND_L5_L10"]), axis=1)

# Final ESPN-adjusted model probability
df["MODEL_PROB_ESPN_V1"] = (df["MODEL_PROB"] + df["ADJ_ROLE"] + df["ADJ_INJ_PROXY"]).clip(0.01, 0.99)

# Top 10 Most Likely (role-verified; dedupe by player)
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V1", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10 = df_rank.head(10).copy()
top10.insert(0, "RANK", range(1, len(top10)+1))

print(top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB","ADJ_ROLE","ADJ_INJ_PROXY","MODEL_PROB_ESPN_V1",
    "MIN_L10","MIN_SD_L10","GAMES_FOUND","MIN_TREND_L5_L10","DNP_RATE_L10"
]].to_string(index=False))

top10.to_csv("nba_prop_engine_v1_top10_most_likely_ESPN_ROLE.csv", index=False)
df.to_csv("nba_prop_engine_v1_full_ESPN_role.csv", index=False)
print("\nSaved: nba_prop_engine_v1_top10_most_likely_ESPN_ROLE.csv")
print("Saved: nba_prop_engine_v1_full_ESPN_role.csv")

In [ ]:
def discord_top10(top10_df):
    lines = ["NBA PROP ENGINE v1 — TOP 10 MOST LIKELY (ESPN ROLE)", "—"]
    for r in top10_df.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME}\n"
            f"Model: {r.MODEL_PROB_ESPN_V1*100:.1f}% | MIN_L10: {r.MIN_L10:.1f} | Trend: {r.MIN_TREND_L5_L10:+.1f}"
        )
    return "\n".join(lines)

print(discord_top10(top10))

In [ ]:
# --- Reduce DVP influence (NBA more efficient than college) ---
def dvp_adjust_nba(p):
    if pd.isna(p): return 0.0
    return float(np.clip(p - 0.50, -0.04, 0.04))

df["ADJ_DVP"] = df["P_DVP_RAW"].apply(dvp_adjust_nba)

df["MODEL_PROB"] = (df["P_TREND"] + df["ADJ_DVP"]).clip(0.01, 0.95)


# --- Stronger Role Penalty ---
def role_adj_v2(min_l10, sd_l10):
    if pd.isna(min_l10):
        base = -0.02
    elif min_l10 >= 35:
        base = +0.035
    elif min_l10 >= 32:
        base = +0.020
    elif min_l10 >= 28:
        base = +0.010
    elif min_l10 >= 24:
        base = -0.010
    elif min_l10 >= 20:
        base = -0.035
    else:
        base = -0.060

    # volatility penalty
    if pd.isna(sd_l10):
        stab = 0.0
    elif sd_l10 >= 8:
        stab = -0.025
    elif sd_l10 >= 6:
        stab = -0.015
    else:
        stab = 0.0

    return base + stab

df["ADJ_ROLE"] = df.apply(lambda r: role_adj_v2(r["MIN_L10"], r["MIN_SD_L10"]), axis=1)

df["MODEL_PROB_ESPN_V2"] = (
    df["MODEL_PROB"] + df["ADJ_ROLE"] + df["ADJ_INJ_PROXY"]
).clip(0.02, 0.90)   # hard NBA realism cap


# Re-rank
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V2", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10_v2 = df_rank.head(10).copy()
top10_v2.insert(0, "RANK", range(1, len(top10_v2)+1))

print(top10_v2[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_ESPN_V2",
    "MIN_L10","MIN_SD_L10","MIN_TREND_L5_L10"
]].to_string(index=False))

In [ ]:
# --- Hard minutes ceiling governor ---
def minutes_ceiling(prob, min_l10):
    if pd.isna(min_l10):
        return min(prob, 0.75)
    if min_l10 < 22:
        return min(prob, 0.70)
    if min_l10 < 24:
        return min(prob, 0.75)
    return prob

df["MODEL_PROB_ESPN_V3"] = df.apply(
    lambda r: minutes_ceiling(r["MODEL_PROB_ESPN_V2"], r["MIN_L10"]),
    axis=1
)

# Re-rank
df_rank = df[df["GAMES_FOUND"].fillna(0) >= 3].copy()
df_rank = df_rank.sort_values("MODEL_PROB_ESPN_V3", ascending=False)
df_rank = df_rank.drop_duplicates(subset=["PLAYER"], keep="first")
top10_v3 = df_rank.head(10).copy()
top10_v3.insert(0, "RANK", range(1, len(top10_v3)+1))

print(top10_v3[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "MODEL_PROB_ESPN_V3",
    "MIN_L10","MIN_SD_L10"
]].to_string(index=False))

In [ ]:
# Recalculate book probabilities
def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

def american_to_decimal(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return 1.0 + (100.0/(-o)) if o < 0 else 1.0 + (o/100.0)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)
df["DEC_ODDS"]  = df["AMERICAN_ODDS"].apply(american_to_decimal)

df["EDGE_VS_BOOK"] = df["MODEL_PROB_ESPN_V3"] - df["BOOK_PROB"]

df["EV_1U"] = df["MODEL_PROB_ESPN_V3"]*(df["DEC_ODDS"]-1) - (1-df["MODEL_PROB_ESPN_V3"])

def kelly(p, d):
    if pd.isna(p) or pd.isna(d): return np.nan
    b = d-1
    q = 1-p
    if b <= 0: return np.nan
    return max(0, (b*p - q)/b)

df["KELLY_HALF"] = 0.5 * df.apply(lambda r: kelly(r["MODEL_PROB_ESPN_V3"], r["DEC_ODDS"]), axis=1)
df["KELLY_HALF_UNITS"] = (df["KELLY_HALF"]*100).round(2)

# Final ranking
final_rank = df[df["GAMES_FOUND"].fillna(0)>=3].copy()
final_rank = final_rank.sort_values("MODEL_PROB_ESPN_V3", ascending=False)
final_rank = final_rank.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10 = final_rank.head(10).copy()
final_top10.insert(0,"RANK",range(1,len(final_top10)+1))

print(final_top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS",
    "MODEL_PROB_ESPN_V3",
    "BOOK_PROB",
    "EDGE_VS_BOOK",
    "KELLY_HALF_UNITS"
]].to_string(index=False))

In [ ]:
def discord_ready(df_block):
    lines = ["NBA PROP ENGINE v1.3 — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.MODEL_PROB_ESPN_V3*100:.1f}% | Edge: {r.EDGE_VS_BOOK*100:+.1f} pts | Half-Kelly: {r.KELLY_HALF_UNITS}u"
        )
    return "\n".join(lines)

print(discord_ready(final_top10))

In [ ]:
def discord_hit_sheet(df_block):
    lines = ["NBA PROP ENGINE — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.MODEL_PROB_CAL*100:.1f}% | Edge: {r.EDGE_CAL*100:+.1f} pts | MIN_L10: {r.MIN_L10:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(final_top10))

In [ ]:
# Ensure book prob exists
def american_to_prob(odds):
    if pd.isna(odds): return np.nan
    o = float(odds)
    return (-o)/((-o)+100.0) if o < 0 else 100.0/(o+100.0)

df["BOOK_PROB"] = df["AMERICAN_ODDS"].apply(american_to_prob)

# Final probability (no compression)
df["FINAL_PROB"] = df["MODEL_PROB_ESPN_V3"]

# Edge
df["FINAL_EDGE"] = df["FINAL_PROB"] - df["BOOK_PROB"]

# Re-rank
final_rank = df[df["GAMES_FOUND"].fillna(0)>=3].copy()
final_rank = final_rank.sort_values("FINAL_PROB", ascending=False)
final_rank = final_rank.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10 = final_rank.head(10).copy()
final_top10.insert(0, "RANK", range(1, len(final_top10)+1))

print(final_top10[[
    "RANK","PLAYER","GAME","PROP","OUTCOME",
    "AMERICAN_ODDS",
    "FINAL_PROB",
    "FINAL_EDGE",
    "MIN_L10"
]].to_string(index=False))

In [ ]:
def discord_hit_sheet(df_block):
    lines = ["NBA PROP ENGINE — TOP 10 MOST LIKELY (ROLE VERIFIED)", "—"]
    for r in df_block.itertuples(index=False):
        lines.append(
            f"{r.RANK}) {r.PLAYER} — {r.GAME}\n"
            f"{r.PROP} {r.OUTCOME} ({int(r.AMERICAN_ODDS)})\n"
            f"Model: {r.FINAL_PROB*100:.1f}% | Edge: {r.FINAL_EDGE*100:+.1f} pts | MIN_L10: {r.MIN_L10:.1f}"
        )
    return "\n".join(lines)

print(discord_hit_sheet(final_top10))

In [ ]:
variance_mask = df["PROP"].str.contains("BLOCK|STEAL", case=False, na=False)
final_rank_novariance = df[(df["GAMES_FOUND"].fillna(0)>=3) & (~variance_mask)].copy()
final_rank_novariance = final_rank_novariance.sort_values("FINAL_PROB", ascending=False)
final_rank_novariance = final_rank_novariance.drop_duplicates(subset=["PLAYER"], keep="first")
final_top10_novariance = final_rank_novariance.head(10).copy()
final_top10_novariance.insert(0, "RANK", range(1, len(final_top10_novariance)+1))

print(discord_hit_sheet(final_top10_novariance))

In [ ]:
import pandas as pd
import numpy as np
import re

CSV_PATH = "NBA 28 NIGHT.csv"  # uploaded file name

df = pd.read_csv(CSV_PATH)

# drop empty index column if present (common in exports)
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", case=False)]
df.columns = [c.strip() for c in df.columns]

# normalize column names to match the notebook
rename_map = {
    "Player": "PLAYER",
    "IM PROB %": "IM_PROB_PCT",
    "HM/AW": "HM_AW",
    "DVP L5": "DVP_L5",
    "DVP L10": "DVP_L10",
    "DVP 2025": "DVP_2025",
    "DVP HM/AW": "DVP_HM_AW",
    "ODDS": "ODDS_RAW",
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

# parse AMERICAN_ODDS if embedded like "BET\n-169"
if "AMERICAN_ODDS" not in df.columns:
    if "ODDS_RAW" in df.columns:
        def extract_american(x):
            if pd.isna(x): return np.nan
            m = re.search(r"([+-]\d{2,5})", str(x))
            return float(m.group(1)) if m else np.nan
        df["AMERICAN_ODDS"] = df["ODDS_RAW"].apply(extract_american)
    else:
        df["AMERICAN_ODDS"] = np.nan

print("Loaded rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(5))

In [ ]:
import os

fname = "NBA 28 NIGHT.csv"
print("cwd:", os.getcwd())
print("exists:", os.path.exists(fname))
print("size bytes:", os.path.getsize(fname) if os.path.exists(fname) else None)
print("dir sample:", os.listdir(".")[:25])